Question4

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mqf_hw1.data import load_data

df = load_data()

In [7]:
d = df.copy()
d = d.set_index(d.columns[0])

d.columns = ["us","intl","bond","bill","cpi"]
real = d[["us", "intl", "bond", "bill"]].add(1).div(1 + d["cpi"], axis=0)- 1


In [8]:
bootstrap_data = real[["us", "intl", "bond", "bill"]].dropna()

print(bootstrap_data.shape)

bootstrap_data.head()

(679, 4)


,us,intl,bond,bill
date,,,,
1970-01-31,-0.077844,-0.023959,0.026105,0.004493
1970-02-28,0.053945,-0.018203,0.062266,0.002681
1970-03-31,-0.002444,-0.006162,-0.005503,0.001805
1970-04-30,-0.095931,-0.112195,-0.058266,-0.002341
1970-05-31,-0.057138,-0.076692,-0.009095,0.003631


(a

In [9]:
def block_bootstrap(data,block_size=24,path_length=120):
    n_months = len(data)
    n_blocks = path_length // block_size
    sampled_blocks = []

    for i in range(n_blocks):
        start = np.random.randint(0,n_months - block_size + 1)
        block = data.iloc[start:start + block_size]
        sampled_blocks.append(block)

    simulation = pd.concat(sampled_blocks,ignore_index=True)

    return simulation


np.random.seed(42)

simulation = block_bootstrap(bootstrap_data)

print(simulation.shape)
simulation.head()



(120, 4)


,us,intl,bond,bill
0,0.048930,0.076899,0.004099,-0.000777
1,0.028943,0.015269,0.013298,0.000290
2,-0.012643,0.021120,-0.011243,-0.002191
3,-0.097711,0.038728,-0.026706,-0.002603
4,0.022201,-0.088133,0.009542,0.002700


(b

In [10]:
np.random.seed(42)

simulations = []

for i in range(1000):
    simulation = block_bootstrap(bootstrap_data)
    simulations.append(simulation)

bootstrap_results = []

for simulation in simulations:

    balanced_return = (0.60 * simulation["us"]+ 0.40 * simulation["bond"])
    balanced_final_wealth = (1 + balanced_return).prod()

    global_return = (0.34 * simulation["us"]+ 0.66 * simulation["intl"])
    global_final_wealth = (1 + global_return).prod()

    bootstrap_results.append({
        "60/40 Balanced": balanced_final_wealth,
        "34/66 Global Equity": global_final_wealth
    })
bootstrap_results = pd.DataFrame(bootstrap_results)

q4_results = pd.DataFrame({
    "Mean": bootstrap_results.mean(),
    "5th Percentile": bootstrap_results.quantile(0.05),
    "Median": bootstrap_results.median(),
    "95th Percentile": bootstrap_results.quantile(0.95)
})

q4_results.round(3)

,Mean,5th Percentile,Median,95th Percentile
60/40 Balanced,1.805,0.907,1.717,2.981
34/66 Global Equity,2.025,0.704,1.776,4.153


(c
As the results above, the "Global Equity portfolio" outperforms the "Balanced stock-bond portfolio" on average performance through the "Mean". However, when we mention downside risk, it seems that "Global Equity portfolio" has larger downside risk than "Balanced portfolio" through the 5th Percentile.

